# Stage 5: submit a Parquet campaign to Fargate

The Parquet equivalent of [3_submit_job.ipynb](./3_submit_job.ipynb). Same compute — AWS Batch on Fargate Spot, the infrastructure that ran the 2025 campaign — but **no DocumentDB anywhere**.

### What changes

| | 3_submit_job (2025) | this notebook |
|---|---|---|
| Work planning | `submit_helper` reads stations from DocumentDB | `shard_planner` writes an immutable queue to S3 |
| One Batch job = | one 40x20 shard | one worker that drains many shards |
| Picks land in | `picks` collection | Parquet on S3 |
| Resume state | `picks_record` collection | `complete/` + `progress/` on S3 |
| Needs a VPC | yes — submission must run inside it | no, runs from anywhere |
| Preemption costs | the whole 800-station-day job | 40 station-day-channels |

### ⚠️ Before running

1. **The image must contain the v3 code.** `ghcr.io/seisscoped/quakescope:latest` is built from `main`; until the v3 PRs merge it has no `pyarrow` and crashes on import, and every task will fail. Check with the cell in §1.
2. **Job queue and compute environment must exist** — see [1_prepare_compute_env.ipynb](./1_prepare_compute_env.ipynb). The 2025 ones (`niyiyu_earthscope`, Fargate Spot, maxvCpus 4000) are reusable.
3. **Cost alerts on** — see [15_monitoring.md](../docs/rerun_2026/15_monitoring.md). A campaign is a multi-day, five-figure action.
4. **Run the smoke test in §5 first.** Not optional: it is how you find out the image, credentials and output prefix all work before committing the full queue.

In [ ]:
import sys, os, json, glob

sys.path.append("..")

import boto3
import pandas as pd

from sb_catalog.src.s3_state import S3CampaignState
from sb_catalog.src.shard_planner import plan, parse_year_day

REGION = "us-east-2"          # where the Fargate Spot quota is
JOB_QUEUE = ""                # e.g. niyiyu_earthscope_missing_station
JOB_DEFINITION = ""           # registered in section 4
IMAGE = "ghcr.io/seisscoped/quakescope:latest"   # PIN A SHORT-SHA FOR A REAL RUN

CAMPAIGN = "s3://<bucket>/<campaign>"            # everything lives under here
WEIGHT = "jma_wc"

batch = boto3.client("batch", region_name=REGION)
assert CAMPAIGN.startswith("s3://") and "<bucket>" not in CAMPAIGN, "set CAMPAIGN"

## 1. Check the image can actually run a Parquet job

This is the failure that wasted a smoke test: the published image imports
`EARTHSCOPE_S3_ACCESS_POINT` unconditionally and has no `pyarrow`, so tasks
exit 1 within seconds and Batch reports them as application failures.

Submits one throwaway task that only checks the imports.

In [ ]:
probe = batch.register_job_definition(
    jobDefinitionName="quakescope_image_probe",
    type="container",
    platformCapabilities=["FARGATE"],
    containerProperties={
        "image": IMAGE,
        # `work --help` reaches the v3 worker through the fixed ENTRYPOINT.
        # It fails on an image that predates the v3 code, which is the point.
        "command": ["work", "--help"],
        "jobRoleArn": f"arn:aws:iam::{boto3.client('sts').get_caller_identity()['Account']}:role/SeisBenchBatchRole",
        "executionRoleArn": f"arn:aws:iam::{boto3.client('sts').get_caller_identity()['Account']}:role/SeisBenchBatchRole",
        "resourceRequirements": [
            {"type": "VCPU", "value": "1"},
            {"type": "MEMORY", "value": "2048"},
        ],
        "networkConfiguration": {"assignPublicIp": "ENABLED"},
        "runtimePlatform": {"operatingSystemFamily": "LINUX", "cpuArchitecture": "X86_64"},
    },
    retryStrategy={"attempts": 1},
    timeout={"attemptDurationSeconds": 600},
)
job = batch.submit_job(jobName="image-probe", jobQueue=JOB_QUEUE,
                       jobDefinition=probe["jobDefinitionName"])
print("probe job:", job["jobId"])
print("SUCCEEDED = the image has the v3 code. FAILED = merge the v3 PRs and let")
print("Actions rebuild, then re-run this cell.")

In [ ]:
import time

while True:
    d = batch.describe_jobs(jobs=[job["jobId"]])["jobs"][0]
    if d["status"] in ("SUCCEEDED", "FAILED"):
        break
    print(d["status"], end="\r")
    time.sleep(10)
print("probe:", d["status"], d.get("statusReason", ""))
assert d["status"] == "SUCCEEDED", "image cannot run the v3 worker - stop here"

## 2. Station metadata to S3

Replaces [2_prepare_station_metadata.ipynb](./2_prepare_station_metadata.ipynb),
which wrote the `stations` collection. Same source files, same `channels`
convention (two-character band codes), written to `stations.parquet` instead.

In [ ]:
frames = []
for netfile in sorted(glob.glob("../networks/*.zip")):
    stations = pd.read_csv(netfile)
    for i, s in stations.iterrows():
        cha = stations.loc[i, "channels"]
        stations.loc[i, "channels"] = ",".join(sorted(set(c[:2] for c in cha.split(","))))
    stations["location_code"] = stations.apply(lambda s: s.id.split(".")[-1], axis=1)
    frames.append(stations)

stations = pd.concat(frames, ignore_index=True)
print(f"{len(stations):,} stations across {stations.network_code.nunique()} networks")

state = S3CampaignState(CAMPAIGN)
state.write_stations(stations)

## 3. Plan the queue

Same grouping as 2025 — 40 stations x 20 days — but written once as an
immutable `shards.jsonl` instead of becoming tens of thousands of
`submit_job` calls.

**The queue is immutable.** Completed work is keyed on shard id, so extending a
campaign's date range means a new campaign prefix, not a rewritten queue.
Check the plan before writing it.

In [ ]:
NETWORK = "CI"                     # comma separated, or None with EXTENT
START, END = "2019.001", "2019.031"

sel = state.get_stations(network=NETWORK)
shards = plan(sel, parse_year_day(START), parse_year_day(END))

station_days = sum(s["n_station_days"] for s in shards)
print(f"{len(sel):,} stations -> {len(shards):,} shards, {station_days:,} station-days")
print(f"at ~54 s per band-day, roughly {station_days * 54 / 3600:,.0f} vCPU-hours of work")
print("\nfirst shard:", json.dumps(shards[0], indent=2)[:300])

In [ ]:
# Writing the queue. Refuses to overwrite an existing one.
state.write_shards(shards)

## 4. Register the worker job definition

One difference from 2025 worth understanding: a job here is **a worker, not a
shard**. It claims shards from the queue until they run out, so you choose how
many workers to run rather than submitting one job per unit of work.

`work` reaches the v3 worker through the image's fixed `ENTRYPOINT`
(`python -m src.picker`), which Batch cannot override.

In [ ]:
account = boto3.client("sts").get_caller_identity()["Account"]
role = f"arn:aws:iam::{account}:role/SeisBenchBatchRole"

jd = batch.register_job_definition(
    jobDefinitionName="quakescope_v3_worker",
    type="container",
    platformCapabilities=["FARGATE"],
    parameters={"campaign": CAMPAIGN, "weight": WEIGHT, "procs": "4"},
    containerProperties={
        "image": IMAGE,
        "command": [
            "work",
            "--campaign", "Ref::campaign",
            "--weight", "Ref::weight",
            "--procs", "Ref::procs",
        ],
        "jobRoleArn": role,          # needs write access to the campaign prefix
        "executionRoleArn": role,
        "resourceRequirements": [
            {"type": "VCPU", "value": "8"},
            {"type": "MEMORY", "value": "16384"},
        ],
        "networkConfiguration": {"assignPublicIp": "ENABLED"},
        "runtimePlatform": {"operatingSystemFamily": "LINUX", "cpuArchitecture": "X86_64"},
        "environment": [
            # Empty is correct for SCEDC/NCEDC. Set both for the EarthScope archive.
            {"name": "EARTHSCOPE_S3_ACCESS_POINT", "value": ""},
            {"name": "ES_OAUTH2__REFRESH_TOKEN", "value": ""},
        ],
    },
    # A preempted worker is replaced; its shard returns to the queue and resumes
    # from its last checkpoint, so retries are cheap and idempotent.
    retryStrategy={"attempts": 10,
                   "evaluateOnExit": [
                       {"action": "RETRY", "onStatusReason": "Your Spot Task was interrupted."},
                       {"action": "EXIT", "onReason": "*"}]},
    timeout={"attemptDurationSeconds": 86400},
)
JOB_DEFINITION = jd["jobDefinitionName"]
print("registered", JOB_DEFINITION, "revision", jd["revision"])

## 5. Smoke test — one worker, a few shards

Do not skip this. It is the cheapest way to find out that the output prefix is
unwritable, the weight name is wrong, or the archive needs credentials — all of
which otherwise surface as thousands of failed tasks.

`--max-shards` stops the worker after a few, leaving the rest of the queue
untouched.

In [ ]:
smoke = batch.submit_job(
    jobName="qs-smoke",
    jobQueue=JOB_QUEUE,
    jobDefinition=JOB_DEFINITION,
    parameters={"campaign": CAMPAIGN, "weight": WEIGHT, "procs": "1"},
    containerOverrides={"command": [
        "work", "--campaign", CAMPAIGN, "--weight", WEIGHT,
        "--procs", "1", "--max-shards", "2", "--profile",
    ]},
)
print("smoke job:", smoke["jobId"])
print("check progress with the cell below, and the picks in 6_check_parquet.ipynb")

In [ ]:
print(state.progress())
d = batch.describe_jobs(jobs=[smoke["jobId"]])["jobs"][0]
print(d["status"], d.get("statusReason", ""))

## 6. Launch the campaign

Only after the smoke test has produced picks you have looked at.

**How many workers?** `N_WORKERS x procs` is the parallelism, and
`N_WORKERS x vCPU` must stay inside both the Fargate Spot vCPU quota and the
compute environment's `maxvCpus` — the smaller of the two wins, and the
compute environment is usually it.

Workers are stateless and idempotent: submit more at any time, and a preempted
one is replaced without anything to clean up.

In [ ]:
N_WORKERS = 20          # x 8 vCPU = 160 vCPU
PROCS = "4"

ce = batch.describe_job_queues(jobQueues=[JOB_QUEUE])["jobQueues"][0][
    "computeEnvironmentOrder"][0]["computeEnvironment"]
cap = batch.describe_compute_environments(computeEnvironments=[ce])[
    "computeEnvironments"][0]["computeResources"]["maxvCpus"]
asked = N_WORKERS * 8
print(f"requesting {asked} vCPU against a compute environment cap of {cap}")
assert asked <= cap, f"raise maxvCpus on {ce} or lower N_WORKERS"

ids = []
for i in range(N_WORKERS):
    r = batch.submit_job(
        jobName=f"qs-worker-{i:03d}",
        jobQueue=JOB_QUEUE,
        jobDefinition=JOB_DEFINITION,
        parameters={"campaign": CAMPAIGN, "weight": WEIGHT, "procs": PROCS},
    )
    ids.append(r["jobId"])
print(f"submitted {len(ids)} workers")
json.dump(ids, open("submitted_workers.json", "w"))

## 7. Stopping

Cancelling a worker does not lose work: its shard returns to the queue and
resumes from its last checkpoint. There is no instance to terminate — Fargate
tasks disappear with the job, which is one of the reasons to prefer it here.

In [ ]:
# for jid in json.load(open("submitted_workers.json")):
#     batch.cancel_job(jobId=jid, reason="operator stop")
# print("cancelled; re-submit section 6 to resume where the queue left off")